In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as F
from itertools import chain

#criacao do dataframe de consulta da tabela bronze.tb_movies_info
df_info = spark.table("bronze.tb_movies_info")    

#renomeação do titulo das colunas 
df_info = (df_info
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

#tratamento da deduplicação mantendo a versão mais recente do filme com base na data de ingestão
deduplicacao = Window.partitionBy("id_filme").orderBy(F.col("ingestion_datetime").desc()) #criação da janela de deduplicação que ordena os dados por data de ingestão

#atualiza o df com a deduplicação dos dados mantendo a versão mais recente do filme com base na data de ingestão 
df_info = (df_info
    .withColumn("row_number", F.row_number().over(deduplicacao))
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

#traducao das colunas
status_filme = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado"
}
#criacao de um mapa de tradução dos status usando o status_filme 
translate_status_map = F.create_map([F.lit(x) for x in chain.from_iterable(status_filme.items())])

#limpeza dos status dos filmes
df_info = (df_info
    #remocao dos hifens e espaços vazios 
    .withColumn("status_limpo", F.lower(F.trim(F.regexp_replace(F.col("status_filme"), r"-", ""))))
    # se bater com o mapa de traducoes, traduzir, se não, retornar "Não Informado"
    .withColumn("status_filme", F.coalesce(translate_status_map[F.col("status_limpo")], F.lit("Não Informado")))
    .drop("status_limpo")
)

#limpeza de minusculos (para que todos os titulos estejam em maiusculo na coluna 'titulo')
df_info = df_info.withColumn("titulo", F.initcap(F.trim(F.col("titulo"))))

#lista de padronização de datas
date_format = ["yyyy-MM-dd", "dd/MM/yyyy", "MM/dd/yyyy", "dd-MM-yyyy", "MM-dd-yyyy"]

#tratamento de datas multi-formato 
#a atualização do dataframe usa a lista date_fmt para tratar os diferentes formatos de datas da tabela movies_info aplicada num loop for, usando o coalesce para retornar a primeira data válida da lista, jogando para o campo ano_lancamento (nova coluna) 
df_info = (df_info
    .withColumn("data_lancamento", F.coalesce(*[F.try_to_date(F.col("data_lancamento"), fmt) for fmt in date_format]))
    .withColumn("ano_lancamento", F.year(F.col("data_lancamento")))
)

#criacao da tabela usando mode overwrite para manter a unicidade da tabela
df_info.write.format("delta").mode("overwrite").saveAsTable("silver.tb_info_filmes")

display(spark.table("silver.tb_info_filmes").limit(10))                                       